# nb_mdf_utils - Metadata-Driven Framework (MDF) shared library

**Do not run this notebook on its own.** Every framework notebook loads it with `%run nb_mdf_utils`.

It holds everything that is *generic*: reading control tables, reading any landed file format,
data-quality rule evaluation, PII de-identification, SCD1/SCD2/insert-only MERGE patterns,
watermarks and run logging. Nothing in here knows about any industry - that all lives in the metadata (config/<pack>.json).

| Layer | Where | What the framework does |
|---|---|---|
| Landing | `Files/landing/<system>/<object>/load_date=YYYY-MM-DD/` | Raw files exactly as received (replayable) |
| Bronze | `bronze.<system>_<object>` | Append per load date, all strings, audit columns, schema drift absorbed |
| Silver | `silver.<entity>` | Typed, renamed, de-duplicated, DQ-checked, PII de-identified, MERGEd (SCD1/SCD2/insert-only) |
| Quarantine | `quarantine.<entity>` | Rows that failed `quarantine` rules - with the failing rule ids; late-arriving rows auto-retried |
| Gold | `gold.<table>` | Star schema + scorecards built from SQL held in `meta.gold_transform` |

In [ ]:
import json, re, uuid, os, traceback
from datetime import datetime, timezone
from pyspark.sql import functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType, LongType, DoubleType, TimestampType,
                               BooleanType, IntegerType)
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable

MDF = {
    "files_rel": "Files",                       # Spark-relative path to the default lakehouse Files area
    "files_local": "/lakehouse/default/Files",  # local mount of the same area (used for listing folders / Excel)
    # Logical layer -> schema. For a multi-lakehouse / multi-workspace production design, change these to
    # three/four-part names, e.g. "lh_bronze.dbo" or "`ws_data_prod`.lh_silver.dbo".
    "schemas": {"meta": "meta", "bronze": "bronze", "silver": "silver", "gold": "gold", "quarantine": "quarantine"},
    # PII salt. In production read it from Key Vault, e.g.
    #   notebookutils.credentials.getSecret("https://<vault>.vault.azure.net/", "pii-salt")
    "pii_salt_default": "demo-salt-change-me",
    "session_timezone": "Australia/Sydney",
}
spark.conf.set("spark.sql.session.timeZone", MDF["session_timezone"])
# lets MERGE ... UPDATE SET * / INSERT * cope with Silver schema evolution (new mapped columns)
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
AUDIT_COLS = ["_source_system", "_object_id", "_load_date", "_ingested_at", "_batch_id", "_source_file"]


def utcnow():
    """Timezone-aware 'now' so Python-written timestamps line up with Spark's current_timestamp()."""
    return datetime.now(timezone.utc)


def tname(layer, table):
    return f"{MDF['schemas'][layer]}.{table}"


def table_exists(name):
    try:
        spark.table(name).limit(0).collect()
        return True
    except AnalysisException:
        return False


def new_run_id(prefix="run"):
    return f"{prefix}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:6]}"


def sanitize(name):
    """Make any source header Delta-safe: 'Clearance (m)' -> 'Clearance_m'."""
    return re.sub(r"[^0-9A-Za-z_]+", "_", str(name).strip()).strip("_") or "col"


def get_setting(key, default=None):
    if not table_exists(tname("meta", "setting")):
        return default
    rows = spark.table(tname("meta", "setting")).where(F.col("key") == key).collect()
    return rows[0]["value"] if rows else default


def pii_salt():
    return get_setting("pii_salt", MDF["pii_salt_default"])


def get_pack():
    """The industry pack definition loaded by nb_mdf_00 (names of the entity, event, reading and party tables)."""
    v = get_setting("pack_spec")
    if not v:
        raise Exception("No pack loaded - run nb_mdf_00_setup_framework first")
    return json.loads(v)


## Control-table access

In [ ]:
def get_objects(object_ids="", load_group="all", only_active=True):
    df = spark.table(tname("meta", "ingestion_object"))
    if only_active:
        df = df.where("active = true")
    ids = [x.strip() for x in (object_ids or "").split(",") if x.strip()]
    if ids:
        df = df.where(F.col("object_id").isin(ids))
    elif load_group and load_group.lower() != "all":
        df = df.where(F.col("load_group").isin([g.strip() for g in load_group.split(",")]))
    return [r.asDict() for r in df.orderBy("priority", "object_id").collect()]


def get_column_map(object_id):
    return [r.asDict() for r in spark.table(tname("meta", "column_map"))
            .where(F.col("object_id") == object_id).orderBy("column_order").collect()]


def get_rules(object_id):
    return [r.asDict() for r in spark.table(tname("meta", "dq_rule"))
            .where((F.col("object_id") == object_id) & (F.col("active") == True)).orderBy("rule_id").collect()]


WATERMARK_SCHEMA = StructType([StructField("object_id", StringType()), StructField("layer", StringType()),
                               StructField("watermark_value", StringType()), StructField("run_id", StringType()),
                               StructField("updated_at", TimestampType())])


def get_watermark(object_id, layer):
    """Watermarks are append-only (no update contention when objects run in parallel); latest row wins."""
    rows = (spark.table(tname("meta", "watermark"))
            .where((F.col("object_id") == object_id) & (F.col("layer") == layer))
            .orderBy(F.col("updated_at").desc()).limit(1).collect())
    return rows[0]["watermark_value"] if rows else None


def set_watermark(object_id, layer, value, run_id):
    if value is None:
        return
    spark.createDataFrame([(object_id, layer, str(value), run_id, utcnow())], WATERMARK_SCHEMA) \
        .write.mode("append").saveAsTable(tname("meta", "watermark"))


RUN_LOG_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("parent_run_id", StringType()),
    StructField("object_id", StringType()), StructField("layer", StringType()), StructField("status", StringType()),
    StructField("started_at", TimestampType()), StructField("ended_at", TimestampType()),
    StructField("duration_sec", DoubleType()), StructField("rows_read", LongType()),
    StructField("rows_written", LongType()), StructField("rows_quarantined", LongType()),
    StructField("rows_deduplicated", LongType()), StructField("message", StringType())])


def log_run(run_id, parent_run_id, object_id, layer, status, started_at, rows_read=0, rows_written=0,
            rows_quarantined=0, rows_deduplicated=0, message=""):
    ended = utcnow()
    row = (run_id, parent_run_id or None, object_id, layer, status, started_at, ended,
           float((ended - started_at).total_seconds()), int(rows_read or 0), int(rows_written or 0),
           int(rows_quarantined or 0), int(rows_deduplicated or 0), (message or "")[:4000])
    spark.createDataFrame([row], RUN_LOG_SCHEMA).write.mode("append").saveAsTable(tname("meta", "run_log"))
    print(f"[{layer}] {object_id}: {status} read={rows_read} written={rows_written} "
          f"quarantined={rows_quarantined} dedup={rows_deduplicated} {('- ' + message) if message else ''}")


DQ_RESULT_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("object_id", StringType()), StructField("rule_id", StringType()),
    StructField("rule_name", StringType()), StructField("rule_category", StringType()),
    StructField("check_type", StringType()), StructField("severity", StringType()),
    StructField("rows_evaluated", LongType()), StructField("rows_failed", LongType()),
    StructField("failed_pct", DoubleType()), StructField("outcome", StringType()),
    StructField("evaluated_at", TimestampType())])

## Reading Landing (any format) - csv / txt / json / xlsx

In [ ]:
def landing_dates(obj):
    """load_date=YYYY-MM-DD folders present in Landing for this object, sorted."""
    local = os.path.join(MDF["files_local"], obj["landing_path"])
    if not os.path.isdir(local):
        return []
    return sorted(d.split("=", 1)[1] for d in os.listdir(local) if d.startswith("load_date="))


def read_landing(obj, dates):
    fmt = obj["landing_format"].lower()
    opts = json.loads(obj.get("read_options") or "{}")
    paths = [f"{MDF['files_rel']}/{obj['landing_path']}/load_date={d}" for d in dates]
    load_date_from_path = F.regexp_extract(F.col("_metadata.file_path"), r"load_date=(\d{4}-\d{2}-\d{2})", 1)

    if fmt in ("csv", "txt"):
        df = (spark.read.option("header", opts.get("header", True)).option("sep", opts.get("delimiter", ","))
              .option("quote", opts.get("quote", '"')).option("escape", '"').option("inferSchema", False)
              .option("encoding", opts.get("encoding", "UTF-8")).csv(paths))   # Bronze = strings (schema-on-read)
        df = df.select("*", F.col("_metadata.file_path").alias("_source_file"), load_date_from_path.alias("_load_date"))

    elif fmt == "json":
        df = spark.read.option("multiLine", opts.get("multiline", True)).json(paths)
        df = df.select("*", F.col("_metadata.file_path").alias("_source_file"), load_date_from_path.alias("_load_date"))
        if opts.get("json_root"):          # e.g. SuccessFactors d.results / Salesforce records / ServiceNow result
            df = df.select(F.explode(F.col(opts["json_root"])).alias("_r"), "_source_file", "_load_date") \
                   .select("_r.*", "_source_file", "_load_date")
        if opts.get("explode"):            # e.g. Ariba lineItems -> one row per line, keeping parent keys
            keep = opts.get("keep_parent", [])
            df = df.select(*keep, "_source_file", "_load_date", F.explode(F.col(opts["explode"])).alias("_x")) \
                   .select(*keep, "_source_file", "_load_date", "_x.*")

    elif fmt in ("xlsx", "xls"):
        import pandas as pd
        frames = []
        for d in dates:
            folder = os.path.join(MDF["files_local"], obj["landing_path"], f"load_date={d}")
            for f in sorted(os.listdir(folder)):
                if f.lower().endswith((".xlsx", ".xls")):
                    pdf = pd.read_excel(os.path.join(folder, f), sheet_name=opts.get("sheet_name", 0),
                                        header=opts.get("header_row", 0), dtype=str)
                    pdf["_source_file"] = f"{obj['landing_path']}/load_date={d}/{f}"
                    pdf["_load_date"] = d
                    frames.append(pdf)
        if not frames:
            raise ValueError(f"No Excel files found for {obj['object_id']} in {dates}")
        pdf = pd.concat(frames, ignore_index=True)
        pdf = pdf.astype(object).where(pdf.notna(), None)
        schema = StructType([StructField(str(c), StringType()) for c in pdf.columns])
        df = spark.createDataFrame(pdf.values.tolist(), schema)
    else:
        raise ValueError(f"Unsupported landing_format '{fmt}' for {obj['object_id']}")

    for c in df.columns:                   # Delta-safe column names
        if c != sanitize(c) and not c.startswith("_"):
            df = df.withColumnRenamed(c, sanitize(c))
    return df

## Data-quality rules (metadata-driven)
Rule types: `not_null`, `regex`, `range`, `allowed_values`, `unique`, `referential`, `expression` (any Spark SQL predicate).
Severities: `warn` (keep row, flag it), `quarantine` (divert row), `fail` (stop the object when failures exceed `fail_threshold_pct`).

In [ ]:
def _flag(rule):
    return "_dq_" + re.sub(r"\W", "_", rule["rule_id"])


def apply_dq_rules(df, rules):
    """Adds one boolean pass-flag column per rule. Returns (df, list_of_flag_columns)."""
    flags = []
    for r in rules:
        p = json.loads(r.get("params") or "{}")
        c = r.get("column_name")
        fc = _flag(r)
        t = r["check_type"]
        if t == "not_null":
            cond = F.col(c).isNotNull()
        elif t == "regex":
            cond = F.col(c).isNull() | F.col(c).cast("string").rlike(p["pattern"])
        elif t == "range":
            cond = F.lit(True)
            if p.get("min") is not None:
                cond = cond & (F.col(c) >= F.lit(p["min"]))
            if p.get("max") is not None:
                cond = cond & (F.col(c) <= F.lit(p["max"]))
            cond = F.col(c).isNull() | cond
        elif t == "allowed_values":
            cond = F.col(c).isNull() | F.col(c).isin(p["values"])
        elif t == "expression":
            cond = F.expr(p["expr"])
        elif t == "unique":
            cols = [x.strip() for x in c.split(",")]
            cond = F.count(F.lit(1)).over(Window.partitionBy(*cols)) == 1
        elif t == "referential":
            ref_key = f"_ref_{fc}"
            if table_exists(p["ref_table"]):
                ref = spark.table(p["ref_table"]).select(F.col(p["ref_column"]).alias(ref_key)).distinct()
                df = df.join(F.broadcast(ref), df[c] == ref[ref_key], "left")
                cond = F.col(c).isNull() | F.col(ref_key).isNotNull()
            else:
                df = df.withColumn(ref_key, F.lit(None).cast("string"))
                cond = F.col(c).isNull()   # reference not loaded yet -> every non-null key fails
        else:
            raise ValueError(f"Unknown check_type {t} in rule {r['rule_id']}")
        df = df.withColumn(fc, F.coalesce(cond, F.lit(False)))
        if t == "referential":
            df = df.drop(f"_ref_{fc}")
        flags.append(fc)
    return df, flags


def summarise_dq(df, rules, run_id, object_id):
    """One pass over the data to count failures per rule; returns list of result tuples."""
    if not rules:
        return [], 0
    aggs = [F.count(F.lit(1)).alias("_n")] + [F.sum(F.when(~F.col(_flag(r)), 1).otherwise(0)).alias(_flag(r)) for r in rules]
    row = df.agg(*aggs).first()
    n = row["_n"]
    out = []
    for r in rules:
        failed = int(row[_flag(r)] or 0)
        pct = round(100.0 * failed / n, 4) if n else 0.0
        if failed == 0:
            outcome = "PASS"
        elif r["severity"] == "fail":
            outcome = "FAILED" if pct > float(r.get("fail_threshold_pct") or 0) else "WARN"
        elif r["severity"] == "quarantine":
            outcome = "QUARANTINED"
        else:
            outcome = "WARN"
        out.append((run_id, object_id, r["rule_id"], r["rule_name"], r["rule_category"], r["check_type"],
                    r["severity"], int(n), failed, pct, outcome, utcnow()))
    return out, n


def write_dq_results(results):
    if results:
        spark.createDataFrame(results, DQ_RESULT_SCHEMA).write.mode("append").saveAsTable(tname("meta", "dq_result"))

## PII de-identification (applied AFTER DQ so rules can see raw values; BEFORE anything is written to Silver/Quarantine)
Treatments: `hash` (salted SHA-256, deterministic so it stays joinable), `mask_email`, `mask_phone`, `year_only`, `redact`, `drop`.

In [ ]:
def apply_pii(df, cmap):
    salt = pii_salt()
    for m in cmap:
        c, t = m["target_column"], (m.get("pii_treatment") or "none")
        if c not in df.columns or t == "none":
            continue
        if t == "hash":
            df = df.withColumn(c, F.when(F.col(c).isNotNull(), F.sha2(F.concat(F.lit(salt), F.col(c).cast("string")), 256)))
        elif t == "mask_email":
            df = df.withColumn(c, F.regexp_replace(F.col(c), r"^(.)[^@]*(@.*)$", "$1***$2"))
        elif t == "mask_phone":
            df = df.withColumn(c, F.when(F.col(c).isNotNull(), F.concat(F.lit("*******"), F.substring(F.col(c), -3, 3))))
        elif t == "year_only":
            df = df.withColumn(c, F.year(F.to_date(F.col(c))).cast("int"))
        elif t == "redact":
            df = df.withColumn(c, F.lit(None).cast("string"))
        elif t == "drop":
            df = df.drop(c)
        else:
            raise ValueError(f"Unknown pii_treatment {t} on {m['object_id']}.{c}")
    return df

## Silver write patterns: SCD1 (with soft delete for full snapshots), SCD2, insert-only

In [ ]:
def ensure_columns(target, df):
    """Schema evolution for Silver: add any new columns from df to the target before MERGE."""
    existing = {f.name.lower() for f in spark.table(target).schema.fields}
    new = [f for f in df.schema.fields if f.name.lower() not in existing]
    if new:
        cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in new)
        spark.sql(f"ALTER TABLE {target} ADD COLUMNS ({cols})")
    return [f.name for f in new]


def _table_version(target):
    return DeltaTable.forName(spark, target).history(1).select("version").first()[0]


def _merge_metrics_since(target, version_before):
    """Empty MERGEs may not commit - only trust metrics from a NEW MERGE commit."""
    h = DeltaTable.forName(spark, target).history(1).select("version", "operation", "operationMetrics").first()
    if h["version"] <= version_before or h["operation"] != "MERGE":
        return {"numTargetRowsInserted": 0, "numTargetRowsUpdated": 0}
    m = h["operationMetrics"] or {}
    return {k: int(v) for k, v in m.items() if k in ("numTargetRowsInserted", "numTargetRowsUpdated",
                                                      "numTargetRowsDeleted", "numOutputRows")}


def record_hash(df, exclude):
    cols = [c for c in df.columns if c not in exclude and not c.startswith("_")]
    return F.sha2(F.concat_ws("||", *[F.coalesce(F.col(f"`{c}`").cast("string"), F.lit("∅")) for c in sorted(cols)]), 256)


def merge_scd1(df, target, keys, full_snapshot=False, snapshot_keys=None):
    df = df.withColumn("_is_deleted", F.lit(False)).withColumn("_silver_updated_at", F.current_timestamp())
    if not table_exists(target):
        df.write.format("delta").saveAsTable(target)
        return {"numTargetRowsInserted": df.count()}
    ensure_columns(target, df)
    v0 = _table_version(target)
    cond = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
    (DeltaTable.forName(spark, target).alias("t").merge(df.alias("s"), cond)
     .whenMatchedUpdateAll(condition="t._record_hash <> s._record_hash OR t._is_deleted = true")
     .whenNotMatchedInsertAll().execute())
    metrics = _merge_metrics_since(target, v0)
    if full_snapshot:
        # Soft delete = keys missing from the WHOLE snapshot (good + quarantined rows), so a row that merely
        # failed a quarantine rule today is not mistaken for a deletion in the source.
        present = (snapshot_keys if snapshot_keys is not None else df).select(*keys).distinct()
        missing = spark.table(target).where("_is_deleted = false").select(*keys).join(present, keys, "left_anti")
        v1 = _table_version(target)
        (DeltaTable.forName(spark, target).alias("t").merge(missing.alias("s"), cond)
         .whenMatchedUpdate(set={"_is_deleted": "true", "_silver_updated_at": "current_timestamp()"}).execute())
        metrics["numSoftDeleted"] = _merge_metrics_since(target, v1).get("numTargetRowsUpdated", 0)
    return metrics


def merge_scd2(df, target, keys, effective_col):
    df = df.withColumn("_valid_from", F.coalesce(F.col(effective_col).cast("timestamp"), F.col("_ingested_at"))) \
           .withColumn("_valid_to", F.lit(None).cast("timestamp")).withColumn("_is_current", F.lit(True)) \
           .withColumn("_silver_updated_at", F.current_timestamp())
    if not table_exists(target):
        df.write.format("delta").saveAsTable(target)
        return {"numTargetRowsInserted": df.count()}
    ensure_columns(target, df)
    v0 = _table_version(target)
    mk = F.concat_ws("||", *[F.col(f"`{k}`").cast("string") for k in keys])
    cur = spark.table(target).where("_is_current = true").select(*keys, F.col("_record_hash").alias("_cur_hash"))
    j = df.withColumn("_mk", mk).join(cur, keys, "left")
    changed = j.where(F.col("_cur_hash").isNull() | (F.col("_cur_hash") != F.col("_record_hash")))
    existing_changed = j.where(F.col("_cur_hash").isNotNull() & (F.col("_cur_hash") != F.col("_record_hash")))
    # classic SCD2 staging: key-matched rows close the current version, null-key rows insert the new version
    staged = changed.withColumn("_merge_key", F.col("_mk")) \
        .unionByName(existing_changed.withColumn("_merge_key", F.lit(None).cast("string"))).drop("_mk", "_cur_hash")
    tcols = [f.name for f in spark.table(target).schema.fields]
    t_mk = "concat_ws('||', " + ", ".join(f"CAST(t.`{k}` AS STRING)" for k in keys) + ")"
    (DeltaTable.forName(spark, target).alias("t")
     .merge(staged.alias("s"), f"{t_mk} = s._merge_key AND t._is_current = true")
     .whenMatchedUpdate(set={"_is_current": "false", "_valid_to": "s._valid_from",
                             "_silver_updated_at": "current_timestamp()"})
     .whenNotMatchedInsert(values={c: f"s.`{c}`" for c in tcols if c in staged.columns})
     .execute())
    return _merge_metrics_since(target, v0)


def insert_only(df, target, keys):
    """Idempotent append for telemetry / events: re-running a file never duplicates readings."""
    if not table_exists(target):
        df.write.format("delta").saveAsTable(target)
        return {"numTargetRowsInserted": df.count()}
    ensure_columns(target, df)
    v0 = _table_version(target)
    cond = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
    DeltaTable.forName(spark, target).alias("t").merge(df.alias("s"), cond).whenNotMatchedInsertAll().execute()
    return _merge_metrics_since(target, v0)


def write_silver(df, obj, keys, is_full_snapshot_batch=True, snapshot_keys=None):
    target = tname("silver", obj["silver_table"])
    mode = obj["silver_write_mode"]
    df = df.withColumn("_record_hash", record_hash(df, keys))
    if mode == "merge_scd1":
        return merge_scd1(df, target, keys, full_snapshot=(obj["load_type"] == "full" and is_full_snapshot_batch),
                          snapshot_keys=snapshot_keys)
    if mode == "merge_scd2":
        eff = [x.strip().split(" ")[0] for x in (obj.get("dedupe_order_by") or "").split(",")][0]
        eff = eff if (eff and eff in df.columns and not eff.startswith("_")) else "_ingested_at"
        return merge_scd2(df, target, keys, eff)
    if mode == "insert_only":
        return insert_only(df, target, keys)
    raise ValueError(f"Unknown silver_write_mode {mode}")

## Helpers used by Bronze -> Silver

In [ ]:
def map_columns(bronze_df, cmap):
    """Rename + type + derive from metadata. Bad casts become NULL (try_cast) and are caught by DQ rules."""
    exprs, used, warnings = [], set(), []
    for m in cmap:
        src, tgt, typ = m["source_column"], m["target_column"], m["data_type"] or "string"
        base = m.get("transform_expr") or f"`{src}`"
        if not m.get("transform_expr") and src not in bronze_df.columns:
            warnings.append(f"missing source column {src}")
            exprs.append(F.lit(None).cast(typ).alias(tgt))
            continue
        used.add(src)
        e = base if (m.get("pii_treatment") == "year_only") else f"try_cast(({base}) AS {typ})"
        exprs.append(F.expr(e).alias(tgt))
    audit = [F.col(c) for c in AUDIT_COLS if c in bronze_df.columns]
    unmapped = [c for c in bronze_df.columns if c not in used and not c.startswith("_") and c != "attributes"]
    return bronze_df.select(*exprs, *audit), unmapped, warnings


def dedupe(df, keys, order_by):
    order = []
    for part in (order_by or "_ingested_at DESC").split(","):
        bits = part.strip().split()
        if not bits or bits[0] not in df.columns:
            continue
        order.append(F.col(bits[0]).desc_nulls_last() if (len(bits) < 2 or bits[1].upper() == "DESC")
                     else F.col(bits[0]).asc_nulls_last())
    order = order or [F.col("_ingested_at").desc()]
    w = Window.partitionBy(*keys).orderBy(*order)
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")


print("nb_mdf_utils loaded - schemas:", MDF["schemas"])